# Lab Cycle 1 — Computational Linguistics Lab
**Course:** 23-813-0704 &nbsp;|&nbsp; **Date:** 16/07/2026

Six problems: regular expressions, an ELIZA-style chatbot, a rule-based tokenizer, a finite-state automaton (FSA), a finite-state transducer (FST), and byte-pair encoding (BPE).

## Problem 1 — Regular Expressions

Four regexes over lines of text. A "word" is taken to mean an alphabetic string, per the assignment's definition.

- **(a)** two consecutive repeated words (`Humbert Humbert`, `the the`, but not `the bug` / `the big bug`)
- **(b)** line starts with an integer and ends with a word
- **(c)** line contains both `grotto` and `raven` as whole words, in either order
- **(d)** first word of a sentence, captured into a register, skipping leading punctuation

In [1]:
import re

# (a) Two consecutive repeated words.
REPEATED_WORD_RE = re.compile(r'\b([A-Za-z]+)\s+\1\b')

# (b) Line starts with an integer, ends with a word (fullmatch => spans whole line).
INT_START_WORD_END_RE = re.compile(r'\d+.*[A-Za-z]+')

# (c) Line contains both "grotto" and "raven" as whole words, in either order.
GROTTO_RAVEN_RE = re.compile(r'(?=.*\bgrotto\b)(?=.*\braven\b)', re.IGNORECASE)

# (d) First word of a sentence -> register, skipping leading whitespace/quotes/brackets.
FIRST_WORD_RE = re.compile(r'^[\s"\'()\[\]]*([A-Za-z]+)')


def find_repeated_words(line):
    return [m.group(0) for m in REPEATED_WORD_RE.finditer(line)]

def starts_int_ends_word(line):
    return INT_START_WORD_END_RE.fullmatch(line) is not None

def has_grotto_and_raven(line):
    return GROTTO_RAVEN_RE.search(line) is not None

def first_word_register(line):
    m = FIRST_WORD_RE.match(line)
    return m.group(1) if m else None


def analyze(lines):
    print("(a) Two consecutive repeated words")
    print("-" * 45)
    found = False
    for ln in lines:
        for hit in find_repeated_words(ln):
            print(f'  MATCH  "{hit}"   <- "{ln}"')
            found = True
    if not found:
        print("  (no matches)")

    print("\n(b) Starts with an integer, ends with a word")
    print("-" * 45)
    found = False
    for ln in lines:
        if starts_int_ends_word(ln):
            print(f'  MATCH  "{ln}"')
            found = True
    if not found:
        print("  (no matches)")

    print("\n(c) Contains both 'grotto' and 'raven'")
    print("-" * 45)
    found = False
    for ln in lines:
        if has_grotto_and_raven(ln):
            print(f'  MATCH  "{ln}"')
            found = True
    if not found:
        print("  (no matches)")

    print("\n(d) First word of each line -> register")
    print("-" * 45)
    for ln in lines:
        reg = first_word_register(ln)
        print(f'  register = {reg!r:<10} <- "{ln}"')


def analyze_file(path):
    """Reads a file and runs the same analysis (per the assignment's file I/O note)."""
    with open(path, "r", encoding="utf-8") as f:
        lines = [ln.rstrip("\n") for ln in f if ln.strip()]
    analyze(lines)


sample_lines = [
    "Humbert Humbert stared out of the window.",
    "I think the the movie was confusing.",
    "the bug crawled across the floor.",
    "the big bug flew away quickly.",
    "42 dogs ran across the yard",
    "3 dogs barked loudly today.",
    "In the misty grotto a raven perched silently.",
    "The grottos near the raven's nest were closed.",
    '"Wait," he shouted, stop right there.',
]

analyze(sample_lines)

(a) Two consecutive repeated words
---------------------------------------------
  MATCH  "Humbert Humbert"   <- "Humbert Humbert stared out of the window."
  MATCH  "the the"   <- "I think the the movie was confusing."

(b) Starts with an integer, ends with a word
---------------------------------------------
  MATCH  "42 dogs ran across the yard"

(c) Contains both 'grotto' and 'raven'
---------------------------------------------
  MATCH  "In the misty grotto a raven perched silently."

(d) First word of each line -> register
---------------------------------------------
  register = 'Humbert'  <- "Humbert Humbert stared out of the window."
  register = 'I'        <- "I think the the movie was confusing."
  register = 'the'      <- "the bug crawled across the floor."
  register = 'the'      <- "the big bug flew away quickly."
  register = None       <- "42 dogs ran across the yard"
  register = None       <- "3 dogs barked loudly today."
  register = 'In'       <- "In the misty grot

## Problem 2 — ELIZA-style chatbot

Domain: **AcadBot**, a lightweight academic advisor. Same substitution mechanics as classic ELIZA — pronoun reflection plus pattern-keyed response templates — applied to statements about courses, exams, and advisors instead of therapy talk. Exits on `BYE BYE`.

In [2]:
import re
import random

random.seed(7)

REFLECTIONS = {
    "i": "you", "me": "you", "my": "your", "mine": "yours",
    "myself": "yourself", "am": "are", "was": "were",
    "i'm": "you're", "i've": "you've", "i'll": "you'll", "i'd": "you'd",
    "you": "I", "your": "my", "yours": "mine",
    "yourself": "myself", "you're": "I'm", "you've": "I've",
    "you'll": "I'll", "you'd": "I'd",
}

def reflect(fragment):
    words = fragment.lower().split()
    return " ".join(REFLECTIONS.get(w, w) for w in words)

RULES = [
    (r"\bi need (.*)", [
        "Why do you need {0}?",
        "Would having {0} really help you?",
    ]),
    (r"\bi want (.*)", [
        "What would it mean to you to get {0}?",
        "Why do you want {0}?",
    ]),
    (r"\bi('?m| am) (.*)", [
        "How long have you been {1}?",
        "Why do you say you are {1}?",
    ]),
    (r"(.*) exam(s)?(.*)", [
        "When is the exam?",
        "How prepared do you feel for it?",
    ]),
    (r"(.*) course(s)?(.*)", [
        "Tell me more about the course.",
        "Which department offers that course?",
    ]),
    (r"(.*)\bbecause\b(.*)", [
        "Is that the real reason?",
        "What other reason might there be?",
    ]),
    (r"(.*)\?$", [
        "Why do you ask that?",
        "What do you think?",
    ]),
    (r"(.*)", [
        "Tell me more about that.",
        "I see. Go on.",
        "Interesting -- can you elaborate?",
        "How does that affect your studies?",
    ]),
]

COMPILED_RULES = [(re.compile(pat, re.IGNORECASE), templates) for pat, templates in RULES]

def respond(user_input):
    for pattern, templates in COMPILED_RULES:
        m = pattern.search(user_input)
        if m:
            groups = [reflect(g) for g in m.groups() if g is not None]
            template = random.choice(templates)
            try:
                return template.format(*groups)
            except (IndexError, KeyError):
                return template
    return "Tell me more."

def run_eliza():
    """Live version -- run this yourself in a fresh cell to actually chat."""
    print("AcadBot: Hi, I'm AcadBot. What's on your mind about classes today? (type BYE BYE to exit)")
    while True:
        user_input = input("You: ")
        if user_input.strip().upper() == "BYE BYE":
            print("AcadBot: Goodbye! Good luck with your studies.")
            break
        print("AcadBot:", respond(user_input))

print("ELIZA rules loaded. Call run_eliza() in a new cell for a live chat, or see the scripted demo below.")

ELIZA rules loaded. Call run_eliza() in a new cell for a live chat, or see the scripted demo below.


In [3]:
# Scripted demo (no live input(), so this cell runs top-to-bottom on its own).
# Call run_eliza() in a fresh cell for a real back-and-forth conversation.
transcript = [
    "I need to register for a new course",
    "I am worried about my thesis deadline",
    "I want to talk to my advisor",
    "Because my advisor is out of town",
    "Do you think I should email the department instead?",
    "BYE BYE",
]

print("AcadBot: Hi, I'm AcadBot. What's on your mind about classes today? (type BYE BYE to exit)")
for line in transcript:
    print("You:", line)
    if line.strip().upper() == "BYE BYE":
        print("AcadBot: Goodbye! Good luck with your studies.")
        break
    print("AcadBot:", respond(line))

AcadBot: Hi, I'm AcadBot. What's on your mind about classes today? (type BYE BYE to exit)
You: I need to register for a new course
AcadBot: Would having to register for a new course really help you?
You: I am worried about my thesis deadline
AcadBot: How long have you been worried about your thesis deadline?
You: I want to talk to my advisor
AcadBot: Why do you want to talk to your advisor?
You: Because my advisor is out of town
AcadBot: Is that the real reason?
You: Do you think I should email the department instead?
AcadBot: Why do you ask that?
You: BYE BYE
AcadBot: Goodbye! Good luck with your studies.


## Problem 3 — Rule-based tokenizer

Punctuation and symbols become their own tokens, `n't`-contractions split per the spec (`isn't` → `is`, `n't`), other contractions split off their suffix (`it's` → `it`, `'s`), abbreviations like `U.S.A.` stay whole, and hyphenated compounds like `ice-cream` stay whole.

In [4]:
import re

TOKEN_PATTERN = re.compile(r"""
    (?P<ABBREV>[A-Za-z](?:\.[A-Za-z])+\.?)
  | (?P<HYPHEN>[A-Za-z]+(?:-[A-Za-z]+)+)
  | (?P<NT>[A-Za-z]+n't)
  | (?P<OTHERC>[A-Za-z]+'(?:re|ve|ll|d|m|s))
  | (?P<WORD>[A-Za-z]+)
  | (?P<NUM>\d+(?:\.\d+)?)
  | (?P<PUNCT>[^\sA-Za-z0-9])
""", re.VERBOSE)

def tokenize(text):
    tokens = []
    for m in TOKEN_PATTERN.finditer(text):
        kind = m.lastgroup
        val = m.group()
        if kind == "NT":                       # isn't -> is / n't
            tokens.append(val[:-3])
            tokens.append("n't")
        elif kind == "OTHERC":                  # it's -> it / 's
            idx = val.index("'")
            tokens.append(val[:idx])
            tokens.append(val[idx:])
        else:
            tokens.append(val)
    return tokens


test_sentence = ("Isn't it true that the U.S.A. has great ice-cream? "
                  "I can't believe it's already 3.5 hours since we've eaten! "
                  "Don't you think so, well-known chef?")

tokens = tokenize(test_sentence)
print("Input: ", test_sentence)
print("\nTokens:")
for t in tokens:
    print(f"  {t!r}")
print(f"\n{len(tokens)} tokens total")

Input:  Isn't it true that the U.S.A. has great ice-cream? I can't believe it's already 3.5 hours since we've eaten! Don't you think so, well-known chef?

Tokens:
  'Is'
  "n't"
  'it'
  'true'
  'that'
  'the'
  'U.S.A.'
  'has'
  'great'
  'ice-cream'
  '?'
  'I'
  'ca'
  "n't"
  'believe'
  'it'
  "'s"
  'already'
  '3.5'
  'hours'
  'since'
  'we'
  "'ve"
  'eaten'
  '!'
  'Do'
  "n't"
  'you'
  'think'
  'so'
  ','
  'well-known'
  'chef'
  '?'

34 tokens total


## Problem 4 — FSA for y-pluralization

Accepts English plurals formed by the two-way `y` rule: **vowel + y → +s** (`boy → boys`) and **consonant + y → +ies** (`pony → ponies`). Rejects the same words pluralized the *wrong* way (`boies`, `ponys`).

Formally: `Q = {q0, qV, qC, qVy, qCi, qCie, qACCEPT1, qACCEPT2, DEAD}`, start state `q0`, accepting states `{qACCEPT1, qACCEPT2}`. Every transition below re-classifies the incoming letter as **V**owel or **C**onsonant whenever it doesn't continue an in-progress match — that "restart" behavior is what lets the automaton work on words of any length rather than one fixed pattern.

In [5]:
VOWELS = set('aeiou')

def delta(state, c):
    c = c.lower()
    is_vowel = c in VOWELS
    is_letter = c.isalpha()

    if state in ('q0', 'qACCEPT1', 'qACCEPT2'):
        if is_vowel:
            return 'qV'
        elif is_letter:
            return 'qC'
        return 'DEAD'

    if state == 'qC':
        if c == 'i':
            return 'qCi'
        elif is_vowel:
            return 'qV'
        elif is_letter:
            return 'qC'
        return 'DEAD'

    if state == 'qV':
        if c == 'y':
            return 'qVy'
        elif is_vowel:
            return 'qV'
        elif is_letter:
            return 'qC'
        return 'DEAD'

    if state == 'qVy':
        if c == 's':
            return 'qACCEPT1'
        elif is_vowel:
            return 'qV'
        elif is_letter:
            return 'qC'
        return 'DEAD'

    if state == 'qCi':
        if c == 'e':
            return 'qCie'
        elif is_vowel:
            return 'qV'
        elif is_letter:
            return 'qC'
        return 'DEAD'

    if state == 'qCie':
        if c == 's':
            return 'qACCEPT2'
        elif is_vowel:
            return 'qV'
        elif is_letter:
            return 'qC'
        return 'DEAD'

    return 'DEAD'

ACCEPTING = {'qACCEPT1', 'qACCEPT2'}

def accepts(word):
    state = 'q0'
    for c in word:
        state = delta(state, c)
        if state == 'DEAD':
            return False
    return state in ACCEPTING


tests = ["boys", "toys", "ponies", "skies", "puppies",   # spec's ACCEPT examples
         "boies", "toies", "ponys",                       # spec's REJECT examples
         "days", "flies", "monkeys", "cities"]             # extra sanity checks

for w in tests:
    print(f"{w:10s} -> {'ACCEPT' if accepts(w) else 'reject'}")

boys       -> ACCEPT
toys       -> ACCEPT
ponies     -> ACCEPT
skies      -> ACCEPT
puppies    -> ACCEPT
boies      -> reject
toies      -> reject
ponys      -> reject
days       -> ACCEPT
flies      -> ACCEPT
monkeys    -> ACCEPT
cities     -> ACCEPT


## Problem 5 — FST for the e-insertion rule

Implements `ε → e / {x,s,z} ^ __ s #`: a silent `e` is inserted between a stem ending in x/s/z and the plural suffix `-s`. Morpheme (`^`) and word (`#`) boundary markers are consumed but never emitted in the surface form.

In [6]:
SIBILANTS = set('xsz')

def transduce(lexical):
    # States: S0 (default), S1 (just saw a sibilant), S2 (boundary after a
    # sibilant -- the insertion site), S4 (boundary after a non-sibilant),
    # FINAL (after the word-boundary marker #).
    state = 'S0'
    output = []
    for ch in lexical:
        if state == 'S0':
            if ch == '^':
                state = 'S4'
            elif ch == '#':
                state = 'FINAL'
            elif ch in SIBILANTS:
                output.append(ch); state = 'S1'
            else:
                output.append(ch)
        elif state == 'S1':
            if ch == '^':
                state = 'S2'
            elif ch == '#':
                state = 'FINAL'
            elif ch in SIBILANTS:
                output.append(ch)
            else:
                output.append(ch); state = 'S0'
        elif state == 'S2':
            if ch == 's':
                output.append('es'); state = 'S0'      # <- the e-insertion
            elif ch == '#':
                state = 'FINAL'
            else:
                output.append(ch); state = 'S0'
        elif state == 'S4':
            if ch == 's':
                output.append('s'); state = 'S0'
            elif ch == '#':
                state = 'FINAL'
            else:
                output.append(ch); state = 'S0'
        elif state == 'FINAL':
            pass
    return ''.join(output)


tests = ["fox^s#", "boy^s#", "bus^s#", "cat^s#", "dog^s#"]
for lex in tests:
    print(f"{lex:10s} -> {transduce(lex)}")

fox^s#     -> foxes
boy^s#     -> boys
bus^s#     -> buses
cat^s#     -> cats
dog^s#     -> dogs


## Problem 6 — Byte Pair Encoding

Toy corpus chosen to make the merges legible: three "low" words and two "new" words sharing prefixes and `-er`/`-est` suffixes, plus `wider` thrown in as a lower-frequency outlier. `_` marks end-of-word so merges never bleed across word boundaries. The loop stops early if the best remaining pair only occurs once — merging a singleton isn't learning a reusable unit, it's memorizing noise.

In [7]:
from collections import Counter

corpus = {
    "low": 5,
    "lower": 2,
    "lowest": 2,
    "newer": 6,
    "newest": 3,
    "wider": 3,
}

END = "_"

def word_to_symbols(word):
    return tuple(list(word) + [END])

vocab = {word_to_symbols(w): freq for w, freq in corpus.items()}

def get_pair_counts(vocab):
    pairs = Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}
    merged_symbol = pair[0] + pair[1]
    for symbols, freq in vocab.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = freq
    return new_vocab

NUM_MERGES = 10
base_vocab = set()
for symbols in vocab:
    base_vocab.update(symbols)

merges = []
print(f"Initial symbol vocabulary ({len(base_vocab)} symbols): {sorted(base_vocab)}\n")

for step in range(1, NUM_MERGES + 1):
    pairs = get_pair_counts(vocab)
    if not pairs:
        print("No more pairs to merge.")
        break
    best_pair = max(pairs, key=pairs.get)
    best_freq = pairs[best_pair]
    if best_freq < 2:
        print(f"Stopping: best remaining pair {best_pair} has frequency {best_freq} < 2")
        break
    vocab = merge_vocab(best_pair, vocab)
    merges.append(best_pair)
    merged_symbol = best_pair[0] + best_pair[1]
    base_vocab.add(merged_symbol)
    print(f"Step {step}: merge {best_pair} -> '{merged_symbol}'  (frequency {best_freq})")
    print(f"  Vocabulary size now: {len(base_vocab)}")
    print("  Current word segmentations:")
    for symbols, freq in vocab.items():
        print(f"    {' '.join(symbols):30s} (freq={freq})")
    print()

print("Final learned merges (in order):")
for i, m in enumerate(merges, 1):
    print(f"  {i}. {m[0]!r} + {m[1]!r} -> {m[0]+m[1]!r}")

Initial symbol vocabulary (11 symbols): ['_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']

Step 1: merge ('w', 'e') -> 'we'  (frequency 13)
  Vocabulary size now: 12
  Current word segmentations:
    l o w _                        (freq=5)
    l o we r _                     (freq=2)
    l o we s t _                   (freq=2)
    n e we r _                     (freq=6)
    n e we s t _                   (freq=3)
    w i d e r _                    (freq=3)

Step 2: merge ('r', '_') -> 'r_'  (frequency 11)
  Vocabulary size now: 13
  Current word segmentations:
    l o w _                        (freq=5)
    l o we r_                      (freq=2)
    l o we s t _                   (freq=2)
    n e we r_                      (freq=6)
    n e we s t _                   (freq=3)
    w i d e r_                     (freq=3)

Step 3: merge ('l', 'o') -> 'lo'  (frequency 9)
  Vocabulary size now: 14
  Current word segmentations:
    lo w _                         (freq=5)
    lo we r_   

In [8]:
def apply_bpe(word, merges):
    symbols = list(word) + [END]
    for pair in merges:
        merged = pair[0] + pair[1]
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

print("Segmenting words the model never trained on, using only the learned merges:\n")
for w in ["slower", "widest", "newly"]:
    print(f"  apply_bpe({w!r}) -> {apply_bpe(w, merges)}")

Segmenting words the model never trained on, using only the learned merges:

  apply_bpe('slower') -> ['s', 'lo', 'we', 'r_']
  apply_bpe('widest') -> ['w', 'i', 'd', 'e', 'st_']
  apply_bpe('newly') -> ['ne', 'w', 'l', 'y', '_']


**Reading the merges above**

The very first merge is `w + e → we` (frequency 13) — not a linguistic suffix at all, just the single most common adjacent symbol pair in the whole corpus. BPE has no notion of morphology; real subword units only emerge as a side effect of counting frequencies, not by design.

Merge order creates path-dependence. `lowest` never becomes `low` + a suffix, even though `low` itself gets merged into its own token at step 8 (`low + _ → low_`). By the time `lo + w → low` is chosen (step 7), the `w` inside `lowest` is already locked inside the earlier `we` merge from step 1, so it's no longer available to combine with `lo`. `lowest` ends up as three tokens (`lo`, `we`, `st_`) instead of the arguably tidier `low` + `est_` — a direct, visible consequence of greedy, one-merge-at-a-time optimization: each step is locally optimal, but the algorithm never revisits earlier choices once made.

Frequency imbalance shows up as uneven compression. `newer` (frequency 6) fully collapses into one token, `newer_`, while `wider` (frequency 3) never merges past `w i d e r_` in the same 10-step budget — it simply never wins a merge round. Higher-frequency words get shorter token sequences "for free"; rarer ones stay closer to raw characters. With a larger merge budget `wider` would eventually compress too, but that's exactly the point: compression rate tracks frequency, not word importance.

That behavior pays off on unseen words. `slower` — never in the training corpus — segments into `s`, `lo`, `we`, `r_`: three of its four tokens are subword pieces learned from `low` and `newer`/`wider`, so the model isn't starting from nothing. `widest` reuses the learned `st_` suffix even though the `wid-` portion of the corpus itself never fully merged. That's the practical case for BPE over a fixed whole-word vocabulary: novel words degrade gracefully into familiar pieces instead of collapsing into a single unknown-word token.

---
All six problems from Lab Cycle 1 done. Every cell above was actually executed — the printed output is real, not hand-written. `run_eliza()` in Problem 2 is the only thing not auto-run (it blocks on `input()`); call it yourself in a new cell for a live back-and-forth.